# AI Video Studio: free LTX-Video server on Google Colab

Runs the open LTX-Video model on Colab's **free GPU** and exposes it as a Gradio API that the
AI Video Studio web app can call. Steps:

1. **Runtime → Change runtime type → T4 GPU** (free), then **Runtime → Run all**.
2. Wait for the last cell to print a `https://xxxx.gradio.live` link (first run downloads about 10 GB of weights; 5–10 minutes).
3. On the phone, open AI Video Studio → **Studio → Footage**, set Generator to *Custom*, paste the link, generate.

The link lives as long as this notebook runs (free Colab sessions last a few hours). Each 5-second clip takes
roughly 2–4 minutes on a T4. Keep this tab open.

In [ ]:
#@title 1. Install (about 2 minutes)
!pip -q install -U "diffusers>=0.33" transformers accelerate sentencepiece imageio imageio-ffmpeg gradio>=5.0 protobuf
import torch, diffusers, gradio
print("torch", torch.__version__, "| diffusers", diffusers.__version__, "| gradio", gradio.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (enable a GPU runtime!)")

In [ ]:
#@title 2. Load LTX-Video (2B, fits the free T4 with CPU offload)
import torch
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

MODEL_ID = "Lightricks/LTX-Video-0.9.5"  #@param ["Lightricks/LTX-Video-0.9.5", "Lightricks/LTX-Video"]
pipe = LTXPipeline.from_pretrained(MODEL_ID, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()          # streams components through the 15 GB T4
pipe.vae.enable_tiling()
print("model ready")

In [ ]:
#@title 3. Start the Gradio API server and print the public link
import random, tempfile, gradio as gr

def text_to_video(prompt, negative_prompt="worst quality, inconsistent motion, blurry, jittery, distorted, text, logo",
                  height=480, width=704, duration=5.0, seed=0, randomize_seed=True, steps=30, guidance=3.0):
    if randomize_seed or not seed:
        seed = random.randint(0, 2**31 - 1)
    height = max(256, int(height) // 32 * 32); width = max(256, int(width) // 32 * 32)
    num_frames = int(max(9, min(161, round(float(duration) * 24 / 8) * 8 + 1)))   # LTX needs 8k+1 frames at 24 fps
    out = pipe(prompt=prompt, negative_prompt=negative_prompt, width=width, height=height, num_frames=num_frames,
               num_inference_steps=int(steps), guidance_scale=float(guidance),
               generator=torch.Generator(device="cuda").manual_seed(int(seed))).frames[0]
    path = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False).name
    export_to_video(out, path, fps=24)
    return path, seed

with gr.Blocks(title="LTX-Video free server for AI Video Studio") as demo:
    gr.Markdown("### LTX-Video server for AI Video Studio\nLeave this running. Paste the public link into Studio → Footage → Generator → Custom.")
    prompt = gr.Textbox(label="Prompt", value="A woman walks through a sunlit office, slow dolly, cinematic")
    negative = gr.Textbox(label="Negative Prompt", value="worst quality, inconsistent motion, blurry, jittery, distorted, text, logo")
    with gr.Row():
        height = gr.Slider(label="Height", minimum=256, maximum=720, step=32, value=480)
        width = gr.Slider(label="Width", minimum=256, maximum=1024, step=32, value=704)
    with gr.Row():
        duration = gr.Slider(label="Video Duration (seconds)", minimum=2, maximum=6.5, step=0.5, value=5)
        steps = gr.Slider(label="Steps", minimum=10, maximum=50, step=1, value=30)
        guidance = gr.Slider(label="Guidance", minimum=1, maximum=8, step=0.5, value=3.0)
    with gr.Row():
        seed = gr.Number(label="Seed", value=0, precision=0)
        randomize = gr.Checkbox(label="Randomize Seed", value=True)
    btn = gr.Button("Generate", variant="primary")
    video = gr.Video(label="Generated Video")
    used_seed = gr.Number(label="Used seed")
    btn.click(text_to_video, inputs=[prompt, negative, height, width, duration, seed, randomize, steps, guidance],
              outputs=[video, used_seed], api_name="text_to_video")

demo.queue(max_size=20).launch(share=True, show_error=True)